In [1]:
import scipy as sp
import numpy as np
import pandas as pd
import pickle

pd.set_option('display.max_rows', None)

In [2]:
def mean_confidence_interval(x, confidence=0.95):
    m, se = np.mean(x), sp.stats.sem(x)
    h = se * sp.stats.t.ppf((1 + confidence) / 2., x.shape[0] - 1)
    return np.round(m - h, 2), np.round(m, 2), np.round(m + h, 2)

def preprocess_string(x):
    if x[0] != "-":
        for i, c in enumerate(x):
            if c.isalpha():
                break
    
        x = x[:i] + " " + x[i:]

        if x[0] == "0" and x[1] != ".":
            i = 1
        else:
            for i in range(1, len(x)):
                if (x[i - 1] == "1"):
                    break
        
        return x[:i] + " " + x[i:]
    else:
        return x

In [3]:
with open("./table_results/diff_results_time_uni_v2.pkl", "rb") as f:
    results = pickle.load(f)

mask_condition = results["mask_condition"]
mask_control = results["mask_control"]
mask_both = results["mask_both"]

In [4]:
keys = [x for x in results.keys() if "mask" not in x]
processed_results = {key: {e: [] for e in ["value", "time"]} for key in keys}

for key in keys:
    tmp_mean = np.mean(results[key]["value"][mask_condition], axis=0)
    processed_results[key]["value"] = mean_confidence_interval(tmp_mean)
    
    processed_results[key]["time"] = mean_confidence_interval(np.array(results[key]["time"]))

In [5]:
df = pd.DataFrame(processed_results).transpose()
df.reset_index(inplace=True)
# df["index"] = df["index"].apply(preprocess_string)

df["sigma_min"] = df["index"].apply(lambda x: x.split()[0])
df["sigma"] = df["index"].apply(lambda x: x.split()[1])
df["score_type"] = df["index"].apply(lambda x: x.split()[2])

df["value"] = df.value.apply(lambda x: np.round(np.array(x), 2))
df["time"] = df.time.apply(lambda x: np.round(np.array(x), 2))

df.drop("index", inplace=True, axis=1)
df = df[["sigma_min", "sigma", "score_type", "value", "time"]]
df.sort_values(["sigma_min", "sigma", "score_type"], ascending=False, inplace=True)
df

,sigma_min,sigma,score_type,value,time
4,0.1,1,rnlvf,"[-13.3, 18.96, 51.22]","[52.75, 54.46, 56.17]"
3,0.1,1,rnl,"[22.63, 23.73, 24.83]","[13.45, 14.27, 15.09]"
2,0.1,1,rl,"[17.84, 18.84, 19.85]","[13.31, 14.37, 15.43]"
1,0.1,1,direct,"[21.3, 22.89, 24.49]","[33.47, 34.59, 35.7]"
24,0.1,0.75,rnlvf,"[-4.95, 17.58, 40.1]","[50.46, 52.43, 54.39]"
23,0.1,0.75,rnl,"[24.18, 25.28, 26.38]","[14.75, 15.61, 16.48]"
22,0.1,0.75,rl,"[19.39, 20.43, 21.47]","[14.74, 15.47, 16.2]"
21,0.1,0.75,direct,"[22.54, 24.19, 25.84]","[37.47, 38.1, 38.74]"
44,0.1,0.5,rnlvf,"[6.77, 10.19, 13.61]","[50.32, 51.8, 53.27]"
43,0.1,0.5,rnl,"[24.75, 26.08, 27.42]","[16.28, 17.51, 18.74]"


In [6]:
group = df.groupby(["score_type"])[["value", "time"]].mean().reset_index()
group["value"] = group.value.apply(lambda x: np.round(np.array(x), 2))
group["time"] = group.time.apply(lambda x: np.round(np.array(x), 2))
group

,score_type,value,time
0,direct,"[20.8, 22.62, 24.45]","[42.09, 43.4, 44.7]"
1,rl,"[18.42, 19.4, 20.37]","[17.82, 19.05, 20.28]"
2,rnl,"[28.38, 31.01, 33.64]","[15.06, 16.38, 17.69]"
3,rnlvf,"[2.63, 10.71, 18.78]","[51.62, 53.99, 56.36]"
4,true,"[20.1, 20.1, 20.1]","[0.01, 0.02, 0.02]"


In [7]:
true = np.array([20.1, 20.1, 20.1])

In [8]:
group = df.copy()
group["value"] = group.value.apply(lambda x: x - true)
group = group.groupby(["sigma_min", "score_type"])[["value", "time"]].mean().reset_index()
group["value"] = group.value.apply(lambda x: np.round(np.array(x), 2))
group["time"] = group.time.apply(lambda x: np.round(np.array(x), 2))
group.sort_values(["sigma_min", "score_type"], ascending=False, inplace=True)
group

,sigma_min,score_type,value,time
20,0.1,rnlvf,"[-19.73, -6.65, 6.43]","[50.13, 51.69, 53.25]"
19,0.1,rnl,"[5.63, 7.61, 9.58]","[15.4, 16.54, 17.69]"
18,0.1,rl,"[-0.96, 0.03, 1.02]","[16.54, 17.7, 18.85]"
17,0.1,direct,"[2.71, 5.33, 7.96]","[40.55, 41.5, 42.46]"
16,0.01,rnlvf,"[-19.98, -12.41, -4.84]","[50.61, 52.95, 55.28]"
15,0.01,rnl,"[8.63, 10.65, 12.67]","[14.95, 16.35, 17.74]"
14,0.01,rl,"[-1.29, -0.28, 0.72]","[17.48, 18.63, 19.78]"
13,0.01,direct,"[1.66, 3.45, 5.24]","[42.98, 44.26, 45.55]"
12,0.001,rnlvf,"[-16.55, -8.98, -1.41]","[52.53, 55.91, 59.28]"
11,0.001,rnl,"[8.54, 11.29, 14.03]","[14.99, 16.38, 17.78]"


In [9]:
group = df.copy()
group["value"] = group.value.apply(lambda x: x - true)
group = group.groupby(["sigma", "score_type"])[["value", "time"]].mean().reset_index()
group["value"] = group.value.apply(lambda x: np.round(np.array(x), 2))
group["time"] = group.time.apply(lambda x: np.round(np.array(x), 2))
group.sort_values(["sigma", "score_type"], ascending=False, inplace=True)
group

,sigma,score_type,value,time
20,1,rnlvf,"[-24.09, -11.4, 1.3]","[60.59, 64.96, 69.34]"
19,1,rnl,"[2.1, 3.52, 4.94]","[13.83, 15.66, 17.5]"
18,1,rl,"[-3.59, -2.7, -1.8]","[14.76, 15.97, 17.19]"
17,1,direct,"[-0.84, 0.67, 2.17]","[36.33, 38.83, 41.31]"
16,0.75,rnlvf,"[-18.67, -6.7, 5.26]","[56.05, 58.42, 60.8]"
15,0.75,rnl,"[3.17, 4.59, 6.02]","[14.9, 15.96, 17.03]"
14,0.75,rl,"[-2.39, -1.49, -0.59]","[16.27, 17.34, 18.41]"
13,0.75,direct,"[0.08, 1.34, 2.6]","[39.2, 39.85, 40.51]"
12,0.5,rnlvf,"[-18.0, -10.86, -3.72]","[53.55, 55.59, 57.62]"
11,0.5,rnl,"[3.47, 5.59, 7.72]","[16.45, 17.66, 18.88]"


In [10]:
df["diff_mean"] = df.value.apply(lambda x: x[1] - true[0])
df["diff_mean_abs"] = df.diff_mean.apply(abs)
df.sort_values("diff_mean_abs")

,sigma_min,sigma,score_type,value,time,diff_mean,diff_mean_abs
0,-,-,true,"[20.1, 20.1, 20.1]","[0.01, 0.02, 0.02]",0.00,0.00
57,0,0.5,direct,"[18.35, 20.08, 21.81]","[42.44, 43.85, 45.27]",-0.02,0.02
70,0.001,0.25,rl,"[18.92, 20.05, 21.17]","[21.28, 22.56, 23.85]",-0.05,0.05
74,0.0001,0.25,rl,"[19.35, 20.23, 21.11]","[19.8, 20.99, 22.18]",0.13,0.13
78,0,0.25,rl,"[19.09, 20.25, 21.4]","[19.74, 21.27, 22.8]",0.15,0.15
9,0.001,1,direct,"[18.35, 20.25, 22.15]","[36.85, 41.43, 46.0]",0.15,0.15
17,0,1,direct,"[18.51, 19.9, 21.28]","[35.71, 37.03, 38.34]",-0.20,0.20
49,0.001,0.5,direct,"[18.61, 19.79, 20.97]","[42.0, 43.2, 44.39]",-0.31,0.31
22,0.1,0.75,rl,"[19.39, 20.43, 21.47]","[14.74, 15.47, 16.2]",0.33,0.33
53,0.0001,0.5,direct,"[18.54, 19.72, 20.91]","[42.7, 43.71, 44.73]",-0.38,0.38
